Display SHAP values in the UI for interpretability by manually computing feature attributions, developing an interactive visualization that allows users to explore individual contributions, and ensuring transparency in model decisions.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import math
import ipywidgets as widgets
import plotly.graph_objects as go

from IPython.display import display, clear_output

In [2]:
df = pd.read_csv("heart_disease_train.csv")

model = joblib.load("saved_models/heart_disease_model_v1.joblib")

print("Dataset loaded:", df.shape)
print("Model loaded successfully!")

Dataset loaded: (79, 19)
Model loaded successfully!


In [13]:
features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

classes = list(model.classes_)

if len(features) != model.n_features_in_:
    raise ValueError("Feature count mismatch!")

print("Features:", features)
print("Feature Count:", len(features))
print("Classes:", classes)

Features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Feature Count: 5
Classes: [np.int64(0), np.int64(1)]


In [14]:
background = df[features].dropna().sample(
    n=min(10, len(df)),
    random_state=42
).reset_index(drop=True)

print("Background records:", len(background))

Background records: 10


In [15]:
def calculate_manual_shap(user_input):

    n = len(features)

    if n > 15:
        raise ValueError("Too many features for exact calculation.")

    x = np.array([user_input[f] for f in features], dtype=object)

    background_array = background.to_numpy()
    total_masks = 2 ** n

    coalition_data = np.tile(
        background_array,
        (total_masks, 1)
    )

    for mask in range(total_masks):
        for j in range(n):
            if mask & (1 << j):
                start = mask * len(background)
                end = start + len(background)
                coalition_data[start:end, j] = x[j]

    coalition_df = pd.DataFrame(
        coalition_data,
        columns=features
    )

    positive_class = classes.index(1) if 1 in classes else len(classes) - 1

    probabilities = model.predict_proba(coalition_df)[:, positive_class]

    coalition_values = probabilities.reshape(
        total_masks, len(background)
    ).mean(axis=1)

    shap_values = []

    for j in range(n):

        contribution = 0

        for mask in range(total_masks):

            if mask & (1 << j):
                continue

            size = bin(mask).count("1")

            weight = (
                math.factorial(size) *
                math.factorial(n - size - 1) /
                math.factorial(n)
            )

            new_mask = mask | (1 << j)

            contribution += weight * (
                coalition_values[new_mask] -
                coalition_values[mask]
            )

        shap_values.append(contribution)

    return np.array(shap_values), coalition_values[0]

In [16]:
input_widgets = {}

for feature in features:

    if pd.api.types.is_numeric_dtype(df[feature]):

        input_widgets[feature] = widgets.FloatText(
            value=float(df[feature].median()),
            description=feature,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="90%")
        )

    else:
        categories = sorted(df[feature].dropna().unique().tolist())

        input_widgets[feature] = widgets.Dropdown(
            options=categories,
            value=categories[0],
            description=feature,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="90%")
        )

display(widgets.VBox(list(input_widgets.values())))

In [ ]:
output = widgets.Output()

def explain_prediction(button):

    with output:
        clear_output(wait=True)

        user_input = {
            feature: input_widgets[feature].value
            for feature in features
        }

        try:
            shap_values, baseline = calculate_manual_shap(user_input)

            input_df = pd.DataFrame([user_input], columns=features)

            probabilities = model.predict_proba(input_df)[0]
            prediction = model.predict(input_df)[0]

            positive_class = classes.index(1) if 1 in classes else len(classes) - 1
            final_probability = probabilities[positive_class]

            print("Predicted Class:", prediction)
            print("Predicted Class Probability:",
                  round(final_probability * 100, 2), "%")

            print("Baseline Probability:",
                  round(baseline * 100, 2), "%")

            print("\nFeature Contributions:")

            for feature, value in zip(features, shap_values):
                print(f"{feature}: {value:+.4f}")

            fig = go.Figure(go.Bar(
                x=shap_values,
                y=features,
                orientation="h",
                marker_color=[
                    "indianred" if value > 0 else "steelblue"
                    for value in shap_values
                ]
            ))

            fig.update_layout(
                title="Manual SHAP Feature Contributions",
                xaxis_title="Contribution to Positive-Class Probability",
                yaxis_title="Features",
                height=max(400, len(features) * 35),
                template="plotly_white"
            )

            fig.add_vline(x=0, line_dash="dash")

            display(fig)

        except Exception as e:
            print("Explanation Error:", e)


explain_button = widgets.Button(
    description="Explain Prediction",
    button_style="success"
)

explain_button.on_click(explain_prediction)

display(explain_button, output)

Button(button_style='success', description='Explain Prediction', style=ButtonStyle())

Output()

In [18]:
feature_selector = widgets.Dropdown(
    options=features,
    description="Select Feature:"
)

detail_output = widgets.Output()

def show_feature_detail(change):

    with detail_output:
        clear_output(wait=True)

        feature = change["new"]
        index = features.index(feature)

        user_input = {
            f: input_widgets[f].value for f in features
        }

        shap_values, baseline = calculate_manual_shap(user_input)

        print("Selected Feature:", feature)
        print("User Input:", user_input[feature])
        print("SHAP Contribution:", round(shap_values[index], 5))

        if shap_values[index] > 0:
            print("Effect: Increases the positive-class probability.")
        elif shap_values[index] < 0:
            print("Effect: Decreases the positive-class probability.")
        else:
            print("Effect: No contribution.")

feature_selector.observe(show_feature_detail, names="value")

display(feature_selector, detail_output)

Dropdown(description='Select Feature:', options=('age', 'trestbps', 'chol', 'thalach', 'oldpeak'), value='age'…

Output()